# Seam Harmonizer v1 Colab Notebook

Готовый ноутбук для **training -> validation -> eval -> export -> verify reload**.

Локально нужно заранее собрать training bundle через `scripts/build_final_training_bundle.py`, загрузить `.tar.gz` на Google Drive и указать путь в параметрах ниже.

In [ ]:
# 0. PARAMS
from pathlib import Path
import os

DATASET_BUNDLE_DRIVE_PATH = '/content/drive/MyDrive/unet_seam/seam_harmonizer_training_bundle.tar.gz'
DRIVE_RUNS_DIR = '/content/drive/MyDrive/unet_seam_runs'
RUN_NAME = 'seam_harmonizer_v1_run001'
REPO_CLONE_URL = 'https://github.com/aaandreyev/unet_seam.git'
REPO_CLONE_REF = 'main'
RUNTIME_ZIP_URL = ''
USE_RAMDISK = True
RAMDISK_SIZE_GB = 140
COPY_ARCHIVE_TO_RAM_FIRST = True
SYNC_INTERVAL_SEC = 60
# 96GB GPU target from spec starts at batch=64; lower this if your Colab GPU is smaller.
TRAIN_BATCH_SIZE = 64
VAL_BATCH_SIZE = 128
TRAIN_EPOCHS = 20
# RTX Blackwell 6000 / high-RAM machines: more workers, larger batches.
TRAIN_NUM_WORKERS = min(4, os.cpu_count() or 2)
PRIMARY_CHECKPOINT = 'best_harmonizer_quality.pt'
PROJECT_ROOT = Path('/content/seam_runtime')
LOCAL_OUTPUTS = PROJECT_ROOT / 'outputs'
LOCAL_CHECKPOINTS = LOCAL_OUTPUTS / 'checkpoints'
LOCAL_EVAL = LOCAL_OUTPUTS / 'eval_reports'
LOCAL_EXPORTS = LOCAL_OUTPUTS / 'exports'
LOCAL_LOGS = LOCAL_OUTPUTS / 'logs'
LOCAL_DATA_ROOT = Path('/content/dataset_bundle')
DRIVE_RUN_DIR = Path(DRIVE_RUNS_DIR) / RUN_NAME
DRIVE_CKPT_DIR = DRIVE_RUN_DIR / 'checkpoints'
DRIVE_EVAL_DIR = DRIVE_RUN_DIR / 'eval_reports'
DRIVE_EXPORT_DIR = DRIVE_RUN_DIR / 'exports'
DRIVE_LOG_DIR = DRIVE_RUN_DIR / 'logs'


In [ ]:
# 1. MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


In [ ]:
# 2. INSTALL / RUNTIME CHECKS
import os, sys, subprocess, importlib.util, platform, json
pkgs = ['pyyaml','scipy','scikit-image','safetensors','tqdm','lpips','psutil']
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'pigz'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
import torch
device = 'cuda' if torch.cuda.is_available() else 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu'
print(json.dumps({'python': sys.executable, 'platform': platform.platform(), 'torch': torch.__version__, 'device': device}, ensure_ascii=False))


In [ ]:
# 3. OPTIONAL RAMDISK
import os, psutil, subprocess, shutil
RAM_ROOT = Path('/content/ramdisk')
if USE_RAMDISK:
    mem = psutil.virtual_memory()
    eff = min(RAMDISK_SIZE_GB, max(4, int(mem.available / (1024**3)) - 22))
    RAM_ROOT.mkdir(parents=True, exist_ok=True)
    mounted = subprocess.run(['mountpoint', str(RAM_ROOT)], capture_output=True).returncode == 0
    if not mounted:
        mount_result = subprocess.run(['mount', '-t', 'tmpfs', '-o', f'size={eff}G,mode=777', 'tmpfs', str(RAM_ROOT)], text=True, capture_output=True)
        if mount_result.returncode != 0:
            print('RAMDISK mount failed; falling back to /content. stderr:', mount_result.stderr[-500:])
            USE_RAMDISK = False
    DATA_ROOT = (RAM_ROOT / 'dataset_bundle') if USE_RAMDISK else LOCAL_DATA_ROOT
else:
    DATA_ROOT = LOCAL_DATA_ROOT
DATA_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUTS.mkdir(parents=True, exist_ok=True)
for p in [LOCAL_CHECKPOINTS, LOCAL_EVAL, LOCAL_EXPORTS, LOCAL_LOGS]:
    p.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)


In [ ]:
# 4. EXTRACT DATASET BUNDLE FROM DRIVE
import shutil, tarfile, time
from tqdm.auto import tqdm

def _safe_extract_member(tf: tarfile.TarFile, member: tarfile.TarInfo, dst: Path) -> None:
    target = (dst / member.name).resolve()
    root = dst.resolve()
    try:
        target.relative_to(root)
    except ValueError as exc:
        raise RuntimeError(f'Unsafe path in tar archive: {member.name!r}') from exc
    if member.isdir():
        target.mkdir(parents=True, exist_ok=True)
        return
    if not member.isfile():
        raise RuntimeError(f'Only regular files/directories are allowed in dataset archive: {member.name!r}')
    src_file = tf.extractfile(member)
    if src_file is None:
        raise RuntimeError(f'Unsafe path in tar archive: {member.name!r}')
    target.parent.mkdir(parents=True, exist_ok=True)
    with src_file, target.open('wb') as out_file:
        shutil.copyfileobj(src_file, out_file, length=8 * 1024 * 1024)

src = Path(DATASET_BUNDLE_DRIVE_PATH)
if not src.exists():
    raise FileNotFoundError(src)
archive_local = src
if COPY_ARCHIVE_TO_RAM_FIRST and USE_RAMDISK:
    archive_local = RAM_ROOT / src.name
    if archive_local.exists() and archive_local.stat().st_size == src.stat().st_size:
        print('archive already copied:', archive_local)
    else:
        with src.open('rb') as fin, archive_local.open('wb') as fout:
            total = src.stat().st_size
            with tqdm(total=total, unit='B', unit_scale=True, desc='copy_archive') as pbar:
                while True:
                    chunk = fin.read(8 * 1024 * 1024)
                    if not chunk:
                        break
                    fout.write(chunk)
                    pbar.update(len(chunk))
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive_local, 'r:*') as tf:
    members = tf.getmembers()
    for m in tqdm(members, desc='extract_bundle', dynamic_ncols=True):
        _safe_extract_member(tf, m, DATA_ROOT)
print('bundle extracted to', DATA_ROOT)


In [ ]:
# 5. FETCH PROJECT FROM GIT OR ZIP (self-contained policy: no embedded repo copy in the notebook)
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path


def _clone_repo() -> bool:
    if not REPO_CLONE_URL:
        return False
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            'git',
            'clone',
            '--depth',
            '1',
            '-b',
            REPO_CLONE_REF,
            REPO_CLONE_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )
    return True


def _unpack_zip() -> None:
    if not RUNTIME_ZIP_URL:
        raise ValueError('RUNTIME_ZIP_URL is empty')
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    zpath = Path('/tmp/unet_seam_runtime_src.zip')
    print('downloading:', RUNTIME_ZIP_URL)
    urllib.request.urlretrieve(RUNTIME_ZIP_URL, zpath)
    tmp = Path('/tmp/unet_seam_src_extract')
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    with zipfile.ZipFile(zpath, 'r') as zf:
        zf.extractall(tmp)
    top = [p for p in tmp.iterdir() if p.is_dir()]
    if len(top) != 1:
        raise RuntimeError('zip must contain exactly one top-level directory, got: ' + repr([p.name for p in top]))
    shutil.move(str(top[0]), str(PROJECT_ROOT))


if _clone_repo():
    pass
elif RUNTIME_ZIP_URL:
    _unpack_zip()
else:
    raise ValueError(
        'Set REPO_CLONE_URL or RUNTIME_ZIP_URL in PARAMS. '
        'This notebook does not embed the repository; code must come from git or a zip URL.'
    )

sys.path.insert(0, str(PROJECT_ROOT))
_n_py = sum(1 for _ in PROJECT_ROOT.rglob('*.py'))
print('PROJECT_ROOT =', PROJECT_ROOT, '| python files =', _n_py)


In [ ]:
# 6. VALIDATE BUNDLE LAYOUT
import json
required = [
    DATA_ROOT / 'manifests/input_raw_manifest.jsonl',
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(p)
print(json.dumps({
    'source_manifest': str(DATA_ROOT / 'manifests/input_raw_manifest.jsonl'),
}, ensure_ascii=False))


In [ ]:
# 7. BUILD RUNTIME CONFIGS (обязателен перед 10, либо 10 сам вызовет тот же скрипт, если yml нет)
import subprocess, sys
subprocess.check_call(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'write_colab_runtime_yamls.py'),
     '--project-root', str(PROJECT_ROOT), '--data-root', str(DATA_ROOT),
     '--train-batch-size', str(TRAIN_BATCH_SIZE), '--val-batch-size', str(VAL_BATCH_SIZE), '--train-epochs', str(TRAIN_EPOCHS),
     '--train-num-workers', str(TRAIN_NUM_WORKERS), '--primary-checkpoint', PRIMARY_CHECKPOINT],
    cwd=str(PROJECT_ROOT),
)
print('runtime configs ->', PROJECT_ROOT / 'runtime_configs')


In [ ]:
# 8. OPTIONAL RESUME FROM DRIVE LAST CHECKPOINT
import shutil
resume_path = None
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
drive_last = DRIVE_CKPT_DIR / 'last.pt'
if drive_last.exists():
    LOCAL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    local_resume = LOCAL_CHECKPOINTS / 'resume_last.pt'
    shutil.copy2(drive_last, local_resume)
    resume_path = local_resume
print('resume_path =', resume_path)


In [ ]:
# 9. BACKGROUND SYNC TO DRIVE (eval/exports on Drive only after you run cells 11–12; checkpoints sync during train if this cell ran before 10)
import threading, time, shutil
if 'SYNC_STOP' in globals():
    SYNC_STOP.set()
SYNC_STOP = threading.Event()
for p in [DRIVE_RUN_DIR, DRIVE_CKPT_DIR, DRIVE_EVAL_DIR, DRIVE_EXPORT_DIR, DRIVE_LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def sync_tree(src: Path, dst: Path) -> int:
    n = 0
    if not src.exists():
        return 0
    for path in src.rglob('*'):
        if not path.is_file():
            continue
        if path.name.startswith('.') or path.suffix in {'.tmp', '.part'}:
            continue
        rel = path.relative_to(src)
        out = dst / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        if not out.exists() or out.stat().st_mtime < path.stat().st_mtime or out.stat().st_size != path.stat().st_size:
            try:
                shutil.copy2(path, out)
                n += 1
            except OSError as e:
                print('sync error', path, '->', out, e, flush=True)
    return n
def sync_all_to_drive() -> None:
    a = sync_tree(LOCAL_CHECKPOINTS, DRIVE_CKPT_DIR)
    b = sync_tree(LOCAL_EVAL, DRIVE_EVAL_DIR)
    c = sync_tree(LOCAL_EXPORTS, DRIVE_EXPORT_DIR)
    d = sync_tree(LOCAL_LOGS, DRIVE_LOG_DIR)
    print('sync tick: ckpt', a, 'eval', b, 'export', c, 'logs', d, '->', DRIVE_RUN_DIR, flush=True)
def sync_loop():
    while True:
        sync_all_to_drive()
        if SYNC_STOP.wait(SYNC_INTERVAL_SEC):
            break
sync_all_to_drive()
sync_thread = threading.Thread(target=sync_loop, daemon=True)
sync_thread.start()
print('background sync started (first tick already ran)')


In [ ]:
# 10b. TensorBoard (можно запускать до или после TRAIN; во время блокирующей ячейки TRAIN новая ячейка Colab не стартует)
import subprocess, sys, time, shutil
from google.colab import output

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "protobuf<5", "tensorboard"], check=True)

LOGDIR = PROJECT_ROOT / "outputs" / "logs" / "tensorboard"
LOGDIR.mkdir(parents=True, exist_ok=True)
subprocess.Popen(
    [sys.executable, "-m", "tensorboard.main", "--logdir", str(LOGDIR), "--port", "6006", "--bind_all"],
    start_new_session=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
DRIVE_TB = DRIVE_RUN_DIR / "tensorboard"
DRIVE_TB.mkdir(parents=True, exist_ok=True)
for path in LOGDIR.rglob("*"):
    if path.is_file():
        rel = path.relative_to(LOGDIR)
        out = DRIVE_TB / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(path, out)
        except OSError:
            pass
print("LOGDIR =", LOGDIR, "| копия на Drive:", DRIVE_TB)
output.serve_kernel_port_as_window(6006, path="/")
print("Если встроенный просмотр серый — открой ссылку в новой вкладке.")

In [ ]:
# 10. TRAIN
import os, sys, subprocess, threading
if not (PROJECT_ROOT / 'runtime_configs' / 'train.yaml').is_file():
    print('Нет runtime_configs — запускаю scripts/write_colab_runtime_yamls.py (как в ячейке 7)…')
    subprocess.check_call(
        [sys.executable, str(PROJECT_ROOT / 'scripts' / 'write_colab_runtime_yamls.py'),
         '--project-root', str(PROJECT_ROOT), '--data-root', str(DATA_ROOT),
         '--train-batch-size', str(TRAIN_BATCH_SIZE), '--val-batch-size', str(VAL_BATCH_SIZE), '--train-epochs', str(TRAIN_EPOCHS),
         '--train-num-workers', str(TRAIN_NUM_WORKERS), '--primary-checkpoint', PRIMARY_CHECKPOINT],
        cwd=str(PROJECT_ROOT),
    )
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'
env['TORCH_CUDNN_V8_API_ENABLED'] = '1'
cmd = [sys.executable, '-u', '-m', 'scripts.train_harmonizer', '--config', str(PROJECT_ROOT / 'runtime_configs/train.yaml')]
resume = globals().get('resume_path')
if resume is not None:
    cmd += ['--resume', str(resume)]
print('TRAIN CMD:', ' '.join(map(str, cmd)))
print('Дальше: loading, train_start, epoch_begin, train_iter_begin (затем пауза до первого train_step — нормально: 1-й батч + воркеры + GPU).')
def _stream_cmd(cmd, cwd, env):
    # Colab: inherited stdio often shows nothing; PIPE + a thread that prints to the cell is reliable. Main thread only p.wait() (not read()).
    p = subprocess.Popen(
        cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    def _pump():
        if p.stdout is not None:
            for line in p.stdout:
                print(line, end='', flush=True)
    t = threading.Thread(target=_pump, daemon=True)
    t.start()
    rc = p.wait()
    t.join(timeout=30)
    if rc != 0:
        print(f'\n[exit {rc}] Причина — в Traceback/ошибке ВЫШЕ. CalledProcessError — только итог.', flush=True)
        raise subprocess.CalledProcessError(rc, cmd)
_stream_cmd(cmd, str(PROJECT_ROOT), env)


In [ ]:
# 11. EVAL
import os, sys, subprocess
if '_stream_cmd' not in globals():
    import threading
    def _stream_cmd(cmd, cwd, env):
        p = subprocess.Popen(
            cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        def _pump():
            if p.stdout is not None:
                for line in p.stdout:
                    print(line, end='', flush=True)
        t = threading.Thread(target=_pump, daemon=True)
        t.start()
        rc = p.wait()
        t.join(timeout=30)
        if rc != 0:
            print(f'\n[exit {rc}] See Traceback above.', flush=True)
            raise subprocess.CalledProcessError(rc, cmd)
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
cmd = [sys.executable, '-u', '-m', 'scripts.run_eval_harmonizer', '--config', str(PROJECT_ROOT / 'runtime_configs/eval.yaml')]
print('EVAL CMD:', ' '.join(map(str, cmd)))
_stream_cmd(cmd, str(PROJECT_ROOT), env)


In [ ]:
# 12. EXPORT + VERIFY
import os, sys, subprocess
if '_stream_cmd' not in globals():
    import threading
    def _stream_cmd(cmd, cwd, env):
        p = subprocess.Popen(
            cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        def _pump():
            if p.stdout is not None:
                for line in p.stdout:
                    print(line, end='', flush=True)
        t = threading.Thread(target=_pump, daemon=True)
        t.start()
        rc = p.wait()
        t.join(timeout=30)
        if rc != 0:
            print(f'\n[exit {rc}] See Traceback above.', flush=True)
            raise subprocess.CalledProcessError(rc, cmd)
env = os.environ.copy()
env['PYTHONPATH'] = str(PROJECT_ROOT)
env['PYTHONUNBUFFERED'] = '1'
export_cmd = [sys.executable, '-u', '-m', 'scripts.export_harmonizer_safetensors', '--config', str(PROJECT_ROOT / 'runtime_configs/export.yaml')]
verify_cmd = [sys.executable, '-u', '-m', 'scripts.verify_harmonizer_export', '--config', str(PROJECT_ROOT / 'runtime_configs/export.yaml')]
print('EXPORT CMD:', ' '.join(map(str, export_cmd)))
_stream_cmd(export_cmd, str(PROJECT_ROOT), env)
print('VERIFY CMD:', ' '.join(map(str, verify_cmd)))
_stream_cmd(verify_cmd, str(PROJECT_ROOT), env)


In [ ]:
# 13. FINAL SYNC + SUMMARY
if 'SYNC_STOP' in globals():
    SYNC_STOP.set()
if 'sync_all_to_drive' in globals():
    sync_all_to_drive()
else:
    print('background sync was not started; skipping final sync')
print('Drive checkpoint files:', sorted(p.name for p in DRIVE_CKPT_DIR.glob('*')))
print('Drive export files:', sorted(p.name for p in DRIVE_EXPORT_DIR.glob('*')))
print('Drive eval runs:', sorted(p.name for p in DRIVE_EVAL_DIR.glob('*')))
